In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import pandas as pd
from config import settings
from services.data_cleaner import DataCleaner

In [13]:
df = pd.read_csv(settings.INPUT_DIR / settings.INPUT_FILE)

In [14]:
df.head()

,transaction_id,lc_number,lc_form,available_with,issue_date,expiry_date,latest_shipment_date,applicant_name,applicant_address,applicant_city,...,incoterm,port_of_loading,port_of_discharge,vessel_name,goods_description,documents_required,additional_conditions,presentation_period_days,payment_terms,charges
0,LC-ZCOTOH,LC-2026-115,STANDBY,BY MIXED PAYMENT,2026-01-11,2026-06-15,2026-05-24,Toyota Motor Corporation,1 Toyota-cho,Toyota City,...,FCA,DEHAM,USSFO,Vessel 15,Office Equipment and Supplies,Commercial Invoice; Packing List; Bill of Lading,NaN,14,AT SIGHT,OUR
1,LC-OH9SDB,LC—2026—148,STANDBY,BY ACCEPTANCE,2026-01-20,2026-05-23,2026-05-05,Apple Inc.,One Apple Park Way,Cupertino,...,FCA,NLRTM,USLAX,Vessel 48,Medical Equipment,Commercial Invoice; Packing List; Bill of Lading,NaN,21,60 DAYS AFTER SIGHT,SHA
2,LC-6GVWRD,LC—2026—138,STANDBY,BY DEFERRED PAYMENT,2026-01-16,2026-07-04,2026-06-14,Toyota Motor Credit,6565 Headquarters Dr,Plano,...,DAP,GBLGP,JPNGO,Vessel 38,Renewable Energy Components,Commercial Invoice; Packing List; Bill of Lading,NaN,14,60 DAYS AFTER SIGHT,SHA
3,LC-0OXFQ8,LC-2026-084,STANDBY,BY MIXED PAYMENT,2026-01-10,2028-06-10,2026-05-23,Samsung Electronics,129 Samsung-ro,Suwon,...,DDP,NLRTM,KRPUS,Vessel 84,Electronic Components,Commercial Invoice; Packing List; Bill of Lading,NaN,21,60 DAYS AFTER SIGHT,OUR
4,LC-Q2Y9V8,LC-2026-069,IRREVOCABLE TRANSFERABLE,BY ACCEPTANCE,2026-01-20,2026-05-25,2026-05-08,Foxconn Technology,No 2 Ziyou St,Tucheng,...,DDP,NaN,TWKHH,Vessel 69,Electronic Components,Commercial Invoice; Packing List; Bill of Lading,NaN,21,90 DAYS AFTER SIGHT,BEN


In [15]:
cleaner = DataCleaner()  #  Creates an instance of the class.
result = cleaner.clean(df)
print(f"Original: {len(df)} rows")
print(f"Cleaned:  {len(result)} rows")


Original: 200 rows
Cleaned:  200 rows


In [16]:
print("BEFORE:", df.loc[1, 'lc_number'])
result = cleaner.clean(df)
print("AFTER: ", result.loc[1, 'lc_number'])

BEFORE: LC—2026—148
AFTER:  LC-2026-148


In [17]:
# Test _clean_case
print("=== CASE NORMALIZATION ===\n")

print("CURRENCY (before → after):")
print("  Before:", df['currency'].value_counts().to_dict())
print("  After: ", result['currency'].value_counts().to_dict())

print("\nSample SWIFT codes:")
for i in [0, 1, 2]:
    print(f"  Row {i}: {df.loc[i, 'issuing_bank_swift']} → {result.loc[i, 'issuing_bank_swift']}")

=== CASE NORMALIZATION ===

CURRENCY (before → after):
  Before: {'JPY': 42, 'USD': 39, 'EUR': 38, 'GBP': 32, 'jpy': 16, 'gbp': 15, 'usd': 12, 'eur': 6}
  After:  {'JPY': 58, 'USD': 51, 'GBP': 47, 'EUR': 44}

Sample SWIFT codes:
  Row 0: SMBCJPJT → SMBCJPJT
  Row 1: WFBIUS6S → WFBIUS6S
  Row 2: CHASUS33 → CHASUS33


# Test _pipeline

In [8]:
# Test_pipeline
from services.validation_service import ValidationService
from services.report_service import ReportService

validator = ValidationService()
results = validator.validate_all(result)  # 'result' = cleaned DataFrame

# Count errors
total_errors = sum(len(errs) for errs in results.values())
flagged = sum(1 for errs in results.values() if errs)
clean = sum(1 for errs in results.values() if not errs)

print(f"Total errors:  {total_errors} (was 92)")
print(f"Flagged:       {flagged} (was 76)")
print(f"Clean:         {clean} (was 124)")

# Show new LC errors specifically
lc_codes = ['LC001', 'LC002', 'LC003', 'LC004', 'LC005', 'LC007', 'LC008']
lc_errors = []
for txn, errs in results.items():
    for e in errs:
        if e.error_code in lc_codes:
            lc_errors.append(e)

print(f"\nNew LC errors:  {len(lc_errors)}")
for e in lc_errors:
    print(f"  {e}")

Total errors:  237 (was 92)
Flagged:       128 (was 76)
Clean:         72 (was 124)

New LC errors:  14
  [CRITICAL] LC005: available_with - Invalid available with option (got: INVALID_OPTION)
  [CRITICAL] LC005: available_with - Invalid available with option (got: INVALID_OPTION)
  [CRITICAL] LC002: lc_number - LC number too short (got: LC)
  [CRITICAL] LC008: confirmation_status - Confirmation status invalid (got: MAYBE)
  [CRITICAL] LC001: lc_number - LC number empty or missing (got: nan)
  [CRITICAL] LC004: lc_form - Invalid LC form type (got: INVALID_FORM)
  [CRITICAL] LC001: lc_number - LC number empty or missing (got: nan)
  [CRITICAL] LC007: confirming_bank_swift - Confirming bank required but missing (got: nan)
  [CRITICAL] LC002: lc_number - LC number too short (got: LC)
  [CRITICAL] LC003: lc_number - LC number too long (got: LC-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX)
  [CRITICAL] LC007: confirming_bank_swift - Confirming bank required but missing (got: nan)
  [CRITICAL] L

In [9]:
print(df['partial_shipment'].value_counts())

partial_shipment
ALLOWED        148
NOT ALLOWED     49
MAYBE            3
Name: count, dtype: int64


In [10]:
from validators.shipment_validator import ShipmentValidator

ship = ShipmentValidator()

# SHIP001: Invalid incoterm
errors = ship.validate(field='incoterm', value='XXX')
print("Invalid incoterm:", errors)

# SHIP002: Port of loading missing
errors = ship.validate(field='incoterm', value='FOB',
                       port_of_loading=None,
                       port_of_discharge='USSFO',
                       partial_shipment='ALLOWED')
print("No loading port (sea):", errors)

# SHIP003: Port of discharge missing
errors = ship.validate(field='incoterm', value='FCA',
                       port_of_loading='DEHAM',
                       port_of_discharge=None,
                       partial_shipment='ALLOWED')
print("No discharge port:", errors)

# SHIP006: Invalid partial shipment
errors = ship.validate(field='incoterm', value='DAP',
                       port_of_loading='DEHAM',
                       port_of_discharge='USSFO',
                       partial_shipment='MAYBE')
print("MAYBE partial:", errors)

# Clean transaction
errors = ship.validate(field='incoterm', value='CIF',
                       port_of_loading='DEHAM',
                       port_of_discharge='USSFO',
                       partial_shipment='ALLOWED')
print("Clean:", errors)

Invalid incoterm: [ValidationError(error_code='SHIP001', message='Invalid incoterm', severity='HIGH', field='incoterm', value='XXX'), ValidationError(error_code='SHIP002', message='Port of loading missing', severity='HIGH', field='port_of_loading', value=None), ValidationError(error_code='SHIP003', message='Port of discharge missing', severity='HIGH', field='port_of_discharge', value=None)]
No loading port (sea): [ValidationError(error_code='SHIP002', message='Port of loading missing', severity='HIGH', field='port_of_loading', value=None), ValidationError(error_code='SHIP005', message='Sea incoterm used but no port specified', severity='HIGH', field='port_of_loading', value='sea incoterm FOB requires port')]
No discharge port: [ValidationError(error_code='SHIP003', message='Port of discharge missing', severity='HIGH', field='port_of_discharge', value=None)]
MAYBE partial: [ValidationError(error_code='SHIP006', message='Partial shipment value invalid', severity='LOW', field='partial_shi

In [18]:
from validators.api.gleif_client import GleifClient

client = GleifClient()
sample_leis = result.loc[result['applicant_lei'].notna(), 'applicant_lei'].head(5)

for lei in sample_leis:
    r = client.lookup(lei)
    print(f"{lei} → {r['status']}")

5493006W3QUS5LMH6R84 → ACTIVE
HWUPKR0MPOU8FGXBT394 → ACTIVE
Z2VZBHUMB7PWWJ63I008 → ACTIVE
529900SECONDLEI12345 → NOT_FOUND
529900SECONDLEI12345 → NOT_FOUND


In [20]:
from collections import Counter

all_errors = []
for errs in results.values():
    for e in errs:
        all_errors.append(e.error_code)

counts = Counter(all_errors)
for code, count in counts.most_common():
    print(f"  {code}: {count}")

  LEI004: 86
  SHIP002: 23
  XVAL002: 20
  SHIP005: 12
  AMT002: 11
  DATE005: 7
  AMT005: 7
  BIC001: 6
  DATE007: 4
  LC005: 4
  AMT006: 4
  XVAL001: 4
  SHIP006: 3
  SHIP004: 3
  LC002: 3
  AMT001: 3
  BIC002: 3
  LEI001: 3
  PTY002: 3
  SHIP001: 3
  DATE002: 2
  LEI002: 2
  LC001: 2
  AMT003: 2
  LC007: 2
  PTY004: 2
  DATE004: 2
  BIC003: 2
  DATE001: 2
  PTY005: 1
  LC008: 1
  LC004: 1
  LC003: 1
  PTY001: 1
  LEI003: 1
  SHIP003: 1
